# Lite3 Hand Pose Registry (Google Colab)

Welcome! This is **Part 1 of 2** in the Lite3 Control using Hand Pose Hands-on Session:
1. **This notebook (`register_pose.ipynb`)**: Teach the computer your custom hand shapes using your webcam and save them into a file called `gestures.json`.
2. **Next notebook (`pose_detection.ipynb`)**: Upload `gestures.json` and use those saved hand shapes to steer the robot dog live!

### What Is a "Pose Class"?
In machine learning, a **class** is simply a named category—just like labeling photo folders `"cat"` and `"dog"`. Here, each class is one hand shape linked to one robot command. During setup, we record one sample pose per class into `gestures.json`:

| Class Name | What the Robot Does | Hand Shape Idea (Pick Clear, Different Poses!) |
|---|---|---|
| `forward` | Walks forward | Open flat palm or 1 finger pointing up |
| `backward` | Walks backward | Thumbs down or fist |
| `left` | Sidesteps left | Thumb pointing left / L-shape |
| `right` | Sidesteps right | Pinky/thumb pointing right |
| `turn_left` | Rotates counter-clockwise | Peace sign (2 fingers) |
| `turn_right` | Rotates clockwise | 3 fingers up |
| `sit` | Robot sits / lies down | OK sign (thumb + index circle) |
| `stand` | Robot stands up | Thumbs up |
| `stop` | Safety stop (all movement speeds set to `0`) | Closed fist or flat stop hand |

> **Beginner Tip**: Pick hand poses that look clearly different from one another (try not to use two poses that only change one pinky angle), and keep your whole hand inside the camera view.


## Block 1: Setup & Installation

### How to Run Cells in Google Colab
To run a grey code box below, hover over it and click the **Play (`▶`)** button on its left side (or click inside it and press **`Shift + Enter`**). Wait until the spinning circle finishes and turns into a green checkmark (`✓`) before moving to the next box.

### What Block 1 Does
1. **Step 1.1 (Install & Download AI Model)**: Installs the Python vision tools (`mediapipe`, `opencv`, `numpy`) and downloads MediaPipe's hand-tracking model (`hand_landmarker.task`), which finds **21 key points (dots)** on your fingers and wrist.
2. **Step 1.2 (Prepare Classes & Pose Math)**:
   - Sets up the **9 `POSE_CLASSES`** listed above.
   - **Normalizes landmarks (`normalize_landmarks`)**: Centers the 21 hand dots on your wrist and scales them by hand size so your pose still matches whether your hand is close to or far from the webcam.
   - **Sets `TOLERANCE` (`0.15`)**: Controls how strict the pose match is (lower = stricter match required; higher = more forgiving).
   - **Defines `GESTURES_PATH` (`gestures.json`)**: The file where your recorded poses are stored. If you already saved a `gestures.json` earlier and want to add/edit poses, upload it in Colab's left **Files** panel before running Block 2.

### Step 1.1: Install Dependencies & Download Hand Landmarker Model


In [ ]:
!pip install -q mediapipe opencv-python-headless numpy

import urllib.request
from pathlib import Path

MODEL_PATH = Path("hand_landmarker.task")
MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/hand_landmarker/"
    "hand_landmarker/float16/1/hand_landmarker.task"
)
if not MODEL_PATH.exists():
    print("Downloading hand_landmarker.task...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
print("Model ready:", MODEL_PATH.resolve())

### Step 1.2: Configure Pose Classes & Load Landmark Helpers

You do not need to edit anything here—just click **Play (`▶`)** to load the 9 `POSE_CLASSES`, default `TOLERANCE` (`0.15`), and helper functions. Look for **`Setup complete. Loaded gestures: [...]`** below the cell when finished (`[]` is normal if you haven't recorded any poses yet).


In [ ]:
import base64
import json
from pathlib import Path

import cv2
import mediapipe as mp
import numpy as np
from mediapipe.tasks.python import vision

try:
    BaseOptions = mp.tasks.BaseOptions
except AttributeError:
    from mediapipe.tasks.python import core
    BaseOptions = core.base_options.BaseOptions

GESTURES_PATH = Path("gestures.json")
TOLERANCE = 0.15
POSE_CLASSES = (
    "forward",
    "backward",
    "left",
    "right",
    "turn_left",
    "turn_right",
    "sit",
    "stand",
    "stop",
)
HAND_CONNECTIONS = (
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (5, 9), (9, 10), (10, 11), (11, 12),
    (9, 13), (13, 14), (14, 15), (15, 16),
    (13, 17), (17, 18), (18, 19), (19, 20),
    (0, 17),
)


def draw_hand_skeleton(frame: np.ndarray, hand_landmarks) -> None:
    if hasattr(vision, "drawing_utils") and hasattr(vision, "drawing_styles") and hasattr(vision, "HandLandmarksConnections"):
        vision.drawing_utils.draw_landmarks(
            frame,
            hand_landmarks,
            vision.HandLandmarksConnections.HAND_CONNECTIONS,
            vision.drawing_styles.get_default_hand_landmarks_style(),
            vision.drawing_styles.get_default_hand_connections_style(),
        )
        return
    h, w = frame.shape[:2]
    pts = [(int(lm.x * w), int(lm.y * h)) for lm in hand_landmarks]
    for a, b in HAND_CONNECTIONS:
        cv2.line(frame, pts[a], pts[b], (0, 220, 0), 2, cv2.LINE_AA)
    for x, y in pts:
        cv2.circle(frame, (x, y), 4, (0, 80, 255), -1, cv2.LINE_AA)


def normalize_landmarks(hand_landmarks) -> np.ndarray:
    coords = np.array([[lm.x, lm.y, lm.z] for lm in hand_landmarks], dtype=np.float32)
    centered = coords - coords[0]
    scale = float(np.max(np.linalg.norm(centered, axis=1)))
    return centered / scale if scale > 1e-6 else centered


def load_gestures(path: Path = GESTURES_PATH) -> dict[str, np.ndarray]:
    if not path.exists():
        return {}
    raw = json.loads(path.read_text(encoding="utf-8"))
    return {name: np.asarray(coords, dtype=np.float32) for name, coords in raw.items()}


def save_gesture(path: Path, name: str, coords: np.ndarray) -> dict[str, np.ndarray]:
    gestures = load_gestures(path)
    gestures[name] = coords.astype(np.float32)
    serializable = {k: np.round(v, 6).tolist() for k, v in gestures.items()}
    path.write_text(json.dumps(serializable, indent=2) + "\n", encoding="utf-8")
    return gestures


def classify_gesture(
    live_coords: np.ndarray,
    gestures: dict[str, np.ndarray],
    tolerance: float = TOLERANCE,
) -> tuple[str, float | None, str | None]:
    if not gestures:
        return "No gestures saved", None, None
    best_name, best_dist = None, float("inf")
    for name, ref_coords in gestures.items():
        if ref_coords.shape != live_coords.shape:
            continue
        dist = float(np.mean(np.linalg.norm(live_coords - ref_coords, axis=1)))
        if dist < best_dist:
            best_dist, best_name = dist, name
    if best_name is None:
        return "No gestures saved", None, None
    if best_dist <= tolerance:
        return best_name, best_dist, best_name
    return "Unknown", best_dist, best_name


def decode_data_url(data_url: str) -> np.ndarray | None:
    if not data_url or "," not in data_url:
        return None
    encoded = data_url.split(",", 1)[1]
    if not encoded:
        return None
    buf = np.frombuffer(base64.b64decode(encoded), dtype=np.uint8)
    if buf.size == 0:
        return None
    return cv2.imdecode(buf, cv2.IMREAD_COLOR)


def encode_data_url(frame: np.ndarray) -> str:
    ok, buf = cv2.imencode(".jpg", frame, [int(cv2.IMWRITE_JPEG_QUALITY), 80])
    if not ok:
        raise RuntimeError("Failed to encode frame")
    return "data:image/jpeg;base64," + base64.b64encode(buf.tobytes()).decode("ascii")


def create_image_landmarker() -> vision.HandLandmarker:
    options = vision.HandLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=str(MODEL_PATH)),
        running_mode=vision.RunningMode.IMAGE,
        num_hands=1,
    )
    return vision.HandLandmarker.create_from_options(options)


print("Setup complete. Loaded gestures:", list(load_gestures().keys()))

## Block 2: Running (Select Class & Save Pose)

Follow these steps to record your personal hand shapes for each class:

1. **Start the camera**: Click **Play (`▶`)** on the cell below, then click **Allow** when your browser asks for webcam permission.
2. **Pick a class**: Click one of the 9 buttons above the video (for example `forward`):
   - **Blue button with yellow border**: Currently selected class ready to save.
   - **Green button with `*`**: Class already has a saved pose inside `gestures.json`.
3. **Save your pose**: Hold that hand pose clearly inside the camera frame until the skeleton lines appear on your hand, then click **SAVE POSE**. You will see `Saved 'forward' (1/9)`.
4. **Repeat & test live recognition**: Record all 9 classes (or at least the commands you plan to use). Once saved, switch hand poses and check the text at the top-left of the video (`Detected: forward (forward: 0.042)`). Smaller error numbers (`err`) mean a closer match!
5. **Finish recording**: Click the red **STOP** button to turn off your webcam, then run the **Export** cell below.

### Troubleshooting
- **Camera blocked / error**: Make sure no other app (Zoom, Teams, or another Colab tab) is using your webcam, then click the camera/lock icon in Chrome's URL bar to allow camera access and re-run the cell.
- **Always says `Unknown` even for a saved pose**: Raise the **Tolerance** box slightly (e.g., from `0.15` to `0.20`), or choose hand poses that look more distinct from each other.
- **Want to replace a bad pose?** Simply click that class's button again, hold your updated hand pose, and click **SAVE POSE**—it overwrites the old sample for that class automatically.


In [ ]:
from google.colab import output
from IPython.display import HTML, JSON, display

_reg_landmarker = create_image_landmarker()
_reg_gestures = load_gestures(GESTURES_PATH)


def _register_frame_cb(data_url: str, target_class: str, do_save: bool, tolerance: float):
    global _reg_gestures
    frame = decode_data_url(data_url)
    if frame is None:
        return JSON({"image": "", "status": "Waiting for valid frame...", "saved": list(_reg_gestures.keys())})

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    res = _reg_landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))

    status = "No hand detected"
    primary_coords = None

    if res.hand_landmarks:
        hand_lms = res.hand_landmarks[0]
        draw_hand_skeleton(frame, hand_lms)
        primary_coords = normalize_landmarks(hand_lms)
        label, dist, closest = classify_gesture(primary_coords, _reg_gestures, float(tolerance))
        color = (0, 220, 0) if label not in ("Unknown", "No gestures saved") else (0, 180, 255)
        text = f"{label}" + (f" ({closest}: {dist:.3f})" if dist is not None else "")
        cv2.putText(frame, text, (16, 36), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2, cv2.LINE_AA)
        status = f"Detected: {text}"

    if do_save:
        if primary_coords is None:
            status = "Cannot save: show your hand in frame first!"
        elif target_class in POSE_CLASSES:
            _reg_gestures = save_gesture(GESTURES_PATH, target_class, primary_coords)
            status = f"Saved '{target_class}' ({len(_reg_gestures)}/{len(POSE_CLASSES)})"

    return JSON({
        "image": encode_data_url(frame),
        "status": status,
        "saved": list(_reg_gestures.keys()),
    })


output.register_callback("register_frame_cb", _register_frame_cb)

buttons_html = "".join(
    f'<button class="cls-btn" data-cls="{c}" onclick="selectClass(this.dataset.cls)" '
    f'style="margin:3px; padding:6px 10px; cursor:pointer;">{c}</button>'
    for c in POSE_CLASSES
)

display(HTML(f"""
<div style="font-family:sans-serif; max-width:720px;">
  <div style="margin-bottom:8px;">
    <strong>Class:</strong> <span id="sel-label" style="color:#0a7;">{POSE_CLASSES[0]}</span>
    &nbsp;|&nbsp;
    <label>Tolerance: <input id="tol" type="number" step="0.01" min="0.01" max="1.0" value="{TOLERANCE}" style="width:65px;"></label>
    &nbsp;|&nbsp;
    <button onclick="requestSave()" style="padding:6px 14px; background:#1976d2; color:#fff; border:none; cursor:pointer; font-weight:bold;">SAVE POSE</button>
    <button onclick="stopRegCam()" style="padding:6px 12px; background:#c62828; color:#fff; border:none; cursor:pointer;">STOP</button>
  </div>
  <div id="cls-bar" style="margin-bottom:8px;">{buttons_html}</div>
  <div id="reg-status" style="margin-bottom:6px; font-weight:bold;">Starting camera...</div>
  <video id="reg-vid" autoplay playsinline style="display:none;"></video>
  <img id="reg-img" style="width:100%; max-width:640px; border:2px solid #444;" />
</div>
<script>
(() => {{
  if (window._robodogStopCam) window._robodogStopCam();
  let selectedClass = "{POSE_CLASSES[0]}";
  let pendingSave = false;
  let running = true;
  let stream = null;
  const video = document.getElementById("reg-vid");
  const img = document.getElementById("reg-img");
  const statusEl = document.getElementById("reg-status");
  const canvas = document.createElement("canvas");
  canvas.width = 640;
  canvas.height = 480;
  const ctx = canvas.getContext("2d");

  window.selectClass = (cls) => {{
    selectedClass = cls;
    document.getElementById("sel-label").textContent = cls;
    refreshButtons([]);
  }};
  window.requestSave = () => {{ pendingSave = true; }};
  const stopFn = () => {{
    running = false;
    if (stream) stream.getTracks().forEach(t => t.stop());
    if (window._robodogStopCam === stopFn) window._robodogStopCam = null;
    statusEl.textContent = "Camera stopped.";
  }};
  window.stopRegCam = stopFn;
  window._robodogStopCam = stopFn;

  let lastSaved = [];
  function refreshButtons(savedList) {{
    if (savedList.length) lastSaved = savedList;
    document.querySelectorAll("#cls-bar .cls-btn").forEach(btn => {{
      const c = btn.dataset.cls;
      const isSel = c === selectedClass;
      const isSaved = lastSaved.includes(c);
      btn.style.background = isSel ? "#0288d1" : (isSaved ? "#2e7d32" : "#424242");
      btn.style.color = "#fff";
      btn.style.border = isSel ? "2px solid #ffeb3b" : "1px solid #888";
      btn.textContent = isSaved ? c + " *" : c;
    }});
  }}
  refreshButtons([]);

  async function loop() {{
    try {{
      stream = await navigator.mediaDevices.getUserMedia({{video: {{width: 640, height: 480}}}});
      video.srcObject = stream;
      await video.play();
      while (running) {{
        if (video.readyState < 2 || video.videoWidth === 0) {{
          await new Promise(r => setTimeout(r, 30));
          continue;
        }}
        ctx.save();
        ctx.translate(canvas.width, 0);
        ctx.scale(-1, 1);
        ctx.drawImage(video, 0, 0, canvas.width, canvas.height);
        ctx.restore();
        const dataUrl = canvas.toDataURL("image/jpeg", 0.8);
        const doSave = pendingSave;
        pendingSave = false;
        const tol = parseFloat(document.getElementById("tol").value) || 0.15;
        const res = await google.colab.kernel.invokeFunction(
          "register_frame_cb", [dataUrl, selectedClass, doSave, tol], {{}}
        );
        let out = res?.data?.["application/json"];
        if (typeof out === "string") out = JSON.parse(out);
        if (out) {{
          if (out.image) img.src = out.image;
          statusEl.textContent = out.status;
          refreshButtons(out.saved || []);
        }}
        await new Promise(r => setTimeout(r, 30));
      }}
    }} catch (err) {{
      statusEl.textContent = "Camera error: " + err;
    }}
  }}
  loop();
}})();
</script>
"""))

### Export: Download `gestures.json`

Click **Play (`▶`)** below to download `gestures.json` into your computer's **`Downloads`** folder. Keep this file handy—you will upload it into Part 2 (`pose_detection.ipynb`) to drive the robot dog (or pass it to the local Python script).


In [ ]:
# Optional: download gestures.json to your computer
from google.colab import files
if GESTURES_PATH.exists():
    files.download(str(GESTURES_PATH))
else:
    print("No gestures.json saved yet.")